# ODMAE Model Explainability Analysis
이 노트북은 학습된 ODMAE 모델의 설명가능성(Explainability)을 체크하기 위해 작성되었습니다.
다음을 확인합니다:
1. 예측에서 중요하게 여겨지는 Static Features + Distance
2. 8개의 Attention Head별 거리 편향(Distance Bias) 역할
3. 주위 OD가 예측에 미치는 영향 (Gate Value)


In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 환경에 맞게 경로 수정
BASE_DIR = '/content/drive/MyDrive/kt/KTDB'
sys.path.insert(0, os.path.join(BASE_DIR, 'src'))

from mae_year.models import ODMAE
from config import STATIC_DATA_23_PATH, MASKING_COLUMNS


In [ ]:
# 1. 데이터 및 모델 로드
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_data_path = os.path.join(BASE_DIR, 'dataset/fixed_eval/fixed_test_dataset.pt')
test_data = torch.load(test_data_path, map_location='cpu')

# 분석할 샘플 선택 (예: 동탄, Task 1)
city = '동탄'
task_id = 1
sample = test_data[city][task_id][0]

N = sample['X_static'].shape[0]
num_features = sample['X_static'].shape[1]

# 모델 초기화 및 가중치 로드
model = ODMAE(num_features=num_features, d_model=128, num_layers=4, nhead=8)
ckpt_path = os.path.join(BASE_DIR, 'best_model/mae_cpc:v5-64epoch.pth')
model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=False)
model.to(device)
model.eval()
print('Model loaded successfully.')


In [ ]:
# 2. Hook 설정 및 Forward Pass (Gradient 계산 포함)

x_static = sample['X_static'].unsqueeze(0).to(device)
x_static.requires_grad_(True)
x_dist = sample['X_dist'].unsqueeze(0).to(device)
x_dist.requires_grad_(True)
x_od_masked = sample['X_OD_masked'].unsqueeze(0).to(device)
A_spatial = sample['A_spatial'].unsqueeze(0).to(device)
mask = sample['mask'].unsqueeze(0).to(device)
active_node_mask = sample.get('active_node_mask', torch.ones_like(mask)).unsqueeze(0).to(device)

activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

model.od_gate.register_forward_hook(get_activation('gate_val'))

with torch.set_grad_enabled(True):
    pred_od = model(x_static, x_od_masked, x_dist, A_spatial, mask, active_node_mask)
    # 마스킹된 노드들의 통행량 합을 기준으로 Gradient 계산
    target_pred = pred_od[0, mask[0], :].sum() + pred_od[0, :, mask[0]].sum()
    target_pred.backward()


### 1. Static Features + Distance Importance

In [ ]:
# Static Features 이름 가져오기
static_df = pd.read_csv(os.path.join(BASE_DIR, 'dataset/final_static_features_2023.csv'))
feature_cols = sorted([c for c in static_df.columns if c not in ['dong_code', 'dong_name']])
feature_cols.extend(['is_masked', 'is_merged'])

# Input * Gradient 방식을 통한 중요도 산출
static_grad = x_static.grad[0].cpu().numpy()
static_val = x_static[0].detach().cpu().numpy()
importance = np.abs(static_grad * static_val).mean(axis=0)

plt.figure(figsize=(10, 6))
sns.barplot(x=importance, y=feature_cols)
plt.title('Static Features Importance (Input x Gradient)')
plt.show()

# Distance Importance
dist_grad = x_dist.grad[0].cpu().numpy()
dist_val = x_dist[0].detach().cpu().numpy()
dist_importance = np.abs(dist_grad * dist_val).mean()
print(f"Distance Feature Overall Importance: {dist_importance:.6f}")


### 2. Attention Head 역할 분석 (8개 Head)
Transformer 모델에서 각 Attention Head가 거리에 따라 어떠한 편향(Bias)을 주는지 분석합니다.

In [ ]:
dist_bias_weights = model.distance_bias.weight.detach().cpu().numpy() # (50, nhead=8)

plt.figure(figsize=(12, 6))
for head in range(dist_bias_weights.shape[1]):
    plt.plot(dist_bias_weights[:, head], label=f'Head {head}')

plt.title('Distance Bias per Attention Head')
plt.xlabel('Distance Bucket (0~50, Log Scaled)')
plt.ylabel('Bias Weight applied to Attention Score')
plt.legend()
plt.show()


### 3. 주위 OD 정보의 영향 (Gate Value)
GCN 및 Attention을 통해 집계된 주변 행정동들의 OD 정보가 현재 행정동의 임베딩에 얼마나 반영되었는지 (0~1)를 나타냅니다.

In [ ]:
gate_val = activations['gate_val'][0].cpu().numpy() # (N, D)
avg_gate_per_node = gate_val.mean(axis=-1) # 차원에 대해 평균 -> (N,)

mask_idx = mask[0].cpu().numpy()
masked_gate = avg_gate_per_node[mask_idx]
unmasked_gate = avg_gate_per_node[~mask_idx]

print(f"평균 Gate Value (전체): {avg_gate_per_node.mean():.4f}")
print(f"마스킹 된 노드(예측 대상)의 평균 Gate Value: {masked_gate.mean():.4f}")
print(f"관측된 노드의 평균 Gate Value: {unmasked_gate.mean():.4f}")

plt.figure(figsize=(8, 5))
sns.histplot(masked_gate, color='red', label='Masked (Target)', alpha=0.5, kde=True)
sns.histplot(unmasked_gate, color='blue', label='Unmasked (Observed)', alpha=0.5, kde=True)
plt.title('Distribution of OD Information Gate Values')
plt.xlabel('Gate Value (0.0 ~ 1.0)')
plt.legend()
plt.show()
